In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv


### Imports & Setup

In [2]:
import os
import re
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
import gc
import warnings
warnings.filterwarnings('ignore')


SEED = 42
TRAIN_FILE = "/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv"
TEST_FILE = "/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv"
SAMPLE_SUB_FILE = "/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv"
OUTPUT_DIR = Path("outputs")

N_FOLDS = 1 # earlier it was 3 but bcz of less time constraint set to 1 
GRAD_ACCUM = 16
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05
MAX_CHUNKS = 16

# --- LEGAL-BERT SETTINGS ---
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN = 512
BATCH_SIZE = 16
LR = 3e-5
EPOCHS = 1 # earlier it was 7 but bcz of less time constraint set to 1 

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Active Model: {MODEL_NAME} | Max Len: {MAX_LEN} | Batch Size: {BATCH_SIZE}")

Active Model: nlpaueb/legal-bert-base-uncased | Max Len: 512 | Batch Size: 16


### Preprocessing


In [3]:
# ── Caching Configuration ──
TRAIN_CACHE = Path("train_clean.csv")
TEST_CACHE = Path("test_clean.csv")
FORCE_RECLEAN = False  # set to True if update the regex logic below

def clean_eu_document(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'^[*+]{2,}\s*\n?', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(\s*\d+\s*\)\s*OJ\b[^\n]+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Having regard to\b.*?(?=\bWhereas\b)', '', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'Having regard to\b.*?(?=\bHAS ADOPTED\b|\bHAS DECIDED\b)', '', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'^THE (?:COUNCIL|COMMISSION|EUROPEAN PARLIAMENT)[^\n]*\n?(?:(?:COMMUNITIES|UNION),\s*\n)?', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^(?:Acting on a proposal|Having consulted|After consulting|After examining)[^\n]*\n?', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'\bDone at\b.+', '', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'\bThis (?:Regulation|Directive|Decision)\b[^\n]*(?:is addressed to|shall be binding)[^\n]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ── Smart Caching Logic ──
if not FORCE_RECLEAN and TRAIN_CACHE.exists() and TEST_CACHE.exists():
    print("Loading pre-processed data from cache (instant)...")
    train_df = pd.read_csv(TRAIN_CACHE)
    test_df = pd.read_csv(TEST_CACHE)
else:
    print("Loading raw data and applying pre-processing regex (this will take a minute)...")
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)
    train_df['clean_text'] = train_df['text'].apply(clean_eu_document)
    test_df['clean_text'] = test_df['text'].apply(clean_eu_document)
    print("Saving clean text to cache...")
    train_df.to_csv(TRAIN_CACHE, index=False)
    test_df.to_csv(TEST_CACHE, index=False)

NUM_CLASSES = train_df['label'].nunique()
print(f"Train shapes: {train_df.shape}, Classes: {NUM_CLASSES}")

class LegalDocDataset(Dataset):
    CPT = 4
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len
        inner = max_len - 2
        self.head_tok = max(1, int(inner * 0.5))
        self.tail_tok = inner - self.head_tok
        self.has_tti = "token_type_ids" in tokenizer("x", return_tensors="pt")

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        text, tok, inner = str(self.texts[idx]), self.tok, self.max_len - 2
        label = int(self.labels[idx]) if self.labels is not None else -1

        if len(text) <= inner * self.CPT:
            enc = tok(text, max_length=self.max_len, truncation=True, padding="max_length", return_tensors="pt")
        else:
            head_ids = tok(text[: self.head_tok * self.CPT], add_special_tokens=False, truncation=True, max_length=10000)["input_ids"][: self.head_tok]
            tail_ids = tok(text[-(self.tail_tok * self.CPT):], add_special_tokens=False, truncation=True, max_length=10000)["input_ids"][-self.tail_tok:] if self.tail_tok > 0 else []
            ids = [tok.cls_token_id] + list(head_ids) + list(tail_ids) + [tok.sep_token_id]
            pad_len = self.max_len - len(ids)
            mask = [1] * len(ids) + [0] * pad_len
            ids += [tok.pad_token_id] * pad_len
            enc = {"input_ids": torch.tensor([ids], dtype=torch.long), "attention_mask": torch.tensor([mask], dtype=torch.long)}

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }
        if self.has_tti:
            item["token_type_ids"] = enc.get("token_type_ids", torch.zeros(1, self.max_len, dtype=torch.long)).squeeze(0)
        return item

def _fwd(batch, device):
    kw = {"input_ids": batch["input_ids"].to(device), "attention_mask": batch["attention_mask"].to(device)}
    if "token_type_ids" in batch: kw["token_type_ids"] = batch["token_type_ids"].to(device)
    return kw

Loading pre-processed data from cache (instant)...
Train shapes: (50840, 4), Classes: 30


### Training

In [5]:
device = torch.device("cuda") # if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

run_name = f"{MODEL_NAME.split('/')[-1]}_{MAX_LEN}_f{N_FOLDS}_s{SEED}"
run_dir = OUTPUT_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

texts_all, labels_all = train_df['clean_text'].values, train_df['label'].values

if N_FOLDS == 1:
    # single stratified hold-out split, for fast iteration
    from sklearn.model_selection import train_test_split
    idx_all = np.arange(len(texts_all))
    t_idx, v_idx = train_test_split(
        idx_all, test_size=0.2, random_state=SEED, stratify=labels_all
    )
    splits = [(t_idx, v_idx)]
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = skf.split(texts_all, labels_all)

for fold, (t_idx, v_idx) in enumerate(splits):
    print(f"\n{'='*40}\n  FOLD {fold+1} / {N_FOLDS}\n{'='*40}")
    fold_dir = run_dir / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = fold_dir / "best_model"
    state_file = fold_dir / "state.json"
    
    if state_file.exists():
        state = json.loads(state_file.read_text())
        if state.get("epochs_trained", 0) >= EPOCHS:
            print(f"Fold {fold} already completed. Skipping.")
            continue
            
    train_ds = LegalDocDataset(texts_all[t_idx], labels_all[t_idx], tokenizer, MAX_LEN)
    val_ds = LegalDocDataset(texts_all[v_idx], labels_all[v_idx], tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True).to(device)
    model.gradient_checkpointing_enable()   # <-- added, trades compute for memory
    loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-6)
    steps_per_epoch = len(train_loader) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(steps_per_epoch * EPOCHS * WARMUP_RATIO), steps_per_epoch * EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    
    best_val_acc = 0.0
    for epoch in range(EPOCHS):
        model.train()
        tr_loss, n_acc = 0.0, 0
        optimizer.zero_grad()
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False)):
            labs = batch["label"].to(device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                logits = model(**_fwd(batch, device)).logits
                loss = loss_fn(logits.float(), labs) / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            tr_loss += loss.item() * GRAD_ACCUM
            n_acc += 1
            
            if n_acc == GRAD_ACCUM or (step + 1 == len(train_loader)):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                n_acc = 0
        
        model.eval()
        vl_loss, preds, trues = 0.0, [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Eval ", leave=False):
                labs = batch["label"].to(device)
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    logits = model(**_fwd(batch, device)).logits
                    l = loss_fn(logits.float(), labs)
                vl_loss += l.item()
                preds.extend(logits.argmax(-1).cpu().numpy())
                trues.extend(labs.cpu().numpy())
        
        vl_acc = accuracy_score(trues, preds)
        is_best = vl_acc > best_val_acc
        print(f"Epoch {epoch+1}: Train Loss {tr_loss/len(train_loader):.4f} | Val Loss {vl_loss/len(val_loader):.4f} | Val Acc {vl_acc:.4f} {'*** BEST' if is_best else ''}")
        
        if is_best:
            best_val_acc = vl_acc
            model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
        state_file.write_text(json.dumps({"best_val_acc": best_val_acc, "epochs_trained": epoch + 1}, indent=2))
        
    del model; gc.collect(); torch.cuda.empty_cache()


  FOLD 1 / 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Epoch 1 Train:   0%|          | 0/2542 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 1 Eval :   0%|          | 0/318 [00:00<?, ?it/s]

Epoch 1: Train Loss 2.0426 | Val Loss 1.5977 | Val Acc 0.6179 *** BEST


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Inference

In [6]:
@torch.no_grad()
def get_chunk_probs(model, tokenizer, texts, max_len, device):
    model.eval()
    chunk_len, stride = max_len - 2, max_len // 2
    results = []
    INFER_BATCH_SIZE = 128
    
    for text in tqdm(texts, desc="Batch Chunk Infer", leave=False):
        token_ids = tokenizer(str(text)[:MAX_CHUNKS * max_len * 4], add_special_tokens=False, truncation=False)["input_ids"]
        
        # 1. Gather all chunks for this specific document first
        all_input_ids, all_attention_masks = [], []
        start = 0
        
        while start < max(len(token_ids), 1):
            chunk = token_ids[start: start + chunk_len]
            if not chunk: break
            
            full = [tokenizer.cls_token_id] + list(chunk) + [tokenizer.sep_token_id]
            pad_len = max_len - len(full)
            att = [1] * len(full) + [0] * pad_len
            full += [tokenizer.pad_token_id] * pad_len
            
            all_input_ids.append(full)
            all_attention_masks.append(att)
            
            if len(all_input_ids) >= MAX_CHUNKS: break
            start += stride
            
        if not all_input_ids:
            results.append(np.zeros((1, NUM_CLASSES)))
            continue
            
        # 2. Fire chunks into the GPU in massive batches
        doc_chunk_probs = []
        for i in range(0, len(all_input_ids), INFER_BATCH_SIZE):
            batch_input_ids = torch.tensor(all_input_ids[i:i+INFER_BATCH_SIZE], dtype=torch.long, device=device)
            batch_attention_mask = torch.tensor(all_attention_masks[i:i+INFER_BATCH_SIZE], dtype=torch.long, device=device)
            
            kwargs = {
                "input_ids": batch_input_ids,
                "attention_mask": batch_attention_mask
            }
            
            with torch.autocast('cuda', dtype=torch.bfloat16):
                logits = model(**kwargs).logits
                
            doc_chunk_probs.append(torch.softmax(logits, -1).cpu().float().numpy())
        
        # 3. Average all chunk probabilities for this single document
        doc_chunk_probs = np.concatenate(doc_chunk_probs, axis=0)
        results.append(np.mean(doc_chunk_probs, axis=0, keepdims=True))
        
    return np.concatenate(results, axis=0)


# INFERENCE MODELS SELECTION
TARGET_INFERENCE_MODELS = [
    "*legal-bert*",
]

device = torch.device("cuda") # if torch.cuda.is_available() else "cpu")
test_texts, test_ids = test_df['text'].values, test_df['id'].values

print(f"Scanning for checkpoints matching: {TARGET_INFERENCE_MODELS}")

model_dirs = set()
for pattern in TARGET_INFERENCE_MODELS:
    model_dirs.update(OUTPUT_DIR.glob(pattern))
    
model_dirs = list(model_dirs)

if not model_dirs:
    print(f"No models found matching {TARGET_INFERENCE_MODELS} in {OUTPUT_DIR}")
else:
    all_probs, weights = [], []
    
    for run_dir in model_dirs:
        if not run_dir.is_dir(): continue
        
        # Extract max_len directly from the directory name (e.g., _512_f3_s42 or head_tail512)
        match = re.search(r'_(\d+)_f\d+_s\d+$', run_dir.name)
        if match:
            run_max_len = int(match.group(1))
        else:
            old_match = re.search(r'(?:head_tail|head)(\d+)', run_dir.name)
            run_max_len = int(old_match.group(1)) if old_match else 512
            
        run_probs, run_val_accs = [], []
        
        for fd in run_dir.glob("fold_*"):
            ckpt, state_file = fd / "best_model", fd / "state.json"
            if not ckpt.exists() or not state_file.exists(): continue
            
            val_acc = json.loads(state_file.read_text()).get("best_val_acc", 0)
            print(f"Loading {fd.name} from {run_dir.name} | Val Acc: {val_acc:.4f} | Max Len: {run_max_len}")
            
            tokenizer = AutoTokenizer.from_pretrained(str(ckpt))
            model = AutoModelForSequenceClassification.from_pretrained(str(ckpt)).to(device)
            
            probs = get_chunk_probs(model, tokenizer, test_texts, run_max_len, device)
            run_probs.append(probs)
            run_val_accs.append(val_acc)
            
            del model; gc.collect(); torch.cuda.empty_cache()
            
        if run_probs:
            avg_run_prob = np.mean(run_probs, axis=0)
            all_probs.append(avg_run_prob)
            run_weight = np.mean(run_val_accs)
            weights.append(run_weight)
            print(f"Run {run_dir.name} Ensemble Weight assigned: {run_weight:.4f}\n")
            
    if all_probs:
        weights = np.array(weights) / np.sum(weights)
        final_probs = np.zeros_like(all_probs[0])
        for prob, w in zip(all_probs, weights):
            final_probs += prob * w
            
        final_preds = final_probs.argmax(axis=1)
        sample_sub = pd.read_csv(SAMPLE_SUB_FILE)
        id_col, label_col = sample_sub.columns[0], sample_sub.columns[1]
        sub_df = pd.DataFrame({id_col: test_ids, label_col: final_preds}).set_index(id_col).loc[sample_sub[id_col]].reset_index()
        sub_df.to_csv("submission.csv", index=False)
        print("Submission saved to submission.csv")
    else:
        print("No valid fold checkpoints were found to infer.")

Scanning for checkpoints matching: ['*legal-bert*']
Loading fold_0 from legal-bert-base-uncased_512_f1_s42 | Val Acc: 0.6179 | Max Len: 512


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batch Chunk Infer:   0%|          | 0/12710 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (563 > 512). Running this sequence through the model will result in indexing errors


Run legal-bert-base-uncased_512_f1_s42 Ensemble Weight assigned: 0.6179

Submission saved to submission.csv


In [7]:
# Grab the probabilities from the FIRST (Legal-BERT) run that was loaded
legal_bert_probs = all_probs[0]

# Convert probabilities to final class predictions
single_model_preds = legal_bert_probs.argmax(axis=1)

# Save to a uniquely named CSV, matching the sample submission's columns/ID order
sample_sub = pd.read_csv(SAMPLE_SUB_FILE)
id_col, label_col = sample_sub.columns[0], sample_sub.columns[1]
sub_df = pd.DataFrame({id_col: test_ids, label_col: single_model_preds}).set_index(id_col).loc[sample_sub[id_col]].reset_index()
sub_df.to_csv("submission.csv", index=False)

print("✅ Success: Standalone Legal-BERT submission saved to submissioncsv")

✅ Success: Standalone Legal-BERT submission saved to submissioncsv
